<a href="https://colab.research.google.com/github/jegankanshika/AI/blob/main/AI_Agent_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from typing import Dict, List, Callable, Any

class Tool:
    def __init__(
        self,
        name: str,
        description: str,
        input_schema: Dict[str, Any],
        output_schema: Dict[str, Any],
        func: Callable[..., Any],
    ):
        self.name = name
        self.description = description
        self.input_schema = input_schema
        self.output_schema = output_schema
        self.func = func
    def __call__(self, **kwargs):
        return self.func(**kwargs)

In [10]:
from typing import Union, Literal
from pydantic import BaseModel

class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Tool] = {}
    def register(self, tool: Tool):
        self.tools[tool.name] = tool
    def get(self, name: str) -> Tool:
        if name not in self.tools.keys():
            raise ValueError(f"Tool '{name}' not found")
        return self.tools[name]
    def list_tools(self) -> List[Dict[str, Any]]:
        return [
            {
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.input_schema.model_json_schema(),
            }
            for tool in self.tools.values()
        ]
    def get_tool_call_args_type(self) -> Union[BaseModel]:
        input_args_models = [tool.input_schema for tool in self.tools.values()]
        tool_call_args = Union[tuple(input_args_models)]
        return tool_call_args
    def get_tool_names(self) -> Literal[None]:
        return Literal[*self.tools.keys()]

In [11]:
[
  {
    "name": "add",
    "description": "Add two numbers",
    "input_schema": {
      "type": "object",
      "properties": {
        "a": {"type": "integer"},
        "b": {"type": "integer"}
      },
      "required": ["a", "b"]
    }
  },
  {
    "name": "multiply",
    "description": "Multiply two numbers",
    "input_schema": {...}
  }
]

[{'name': 'add',
  'description': 'Add two numbers',
  'input_schema': {'type': 'object',
   'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}},
   'required': ['a', 'b']}},
 {'name': 'multiply',
  'description': 'Multiply two numbers',
  'input_schema': {Ellipsis}}]

In [12]:
[
  {
    "name": "add",
    "description": "Add two numbers",
    "input_schema": {
      "type": "object",
      "properties": {
        "a": {"type": "integer"},
        "b": {"type": "integer"}
      },
      "required": ["a", "b"]
    }
  },
  {
    "name": "multiply",
    "description": "Multiply two numbers",
    "input_schema": {...}
  }
]

[{'name': 'add',
  'description': 'Add two numbers',
  'input_schema': {'type': 'object',
   'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}},
   'required': ['a', 'b']}},
 {'name': 'multiply',
  'description': 'Multiply two numbers',
  'input_schema': {Ellipsis}}]

In [13]:
class ToolAddArgs(BaseModel):
    a: int
    b: int

class ToolMultiplyArgs(BaseModel):
    a: int
    b: int

In [14]:
def add(a: int, b: int) -> int:
    return a + b

def multiply(a: int, b: int) -> int:
    return a * b

registry = ToolRegistry()

registry.register(
    Tool(
        name="add",
        description="Add two numbers",
        input_schema=ToolAddArgs,
        output_schema={"result": "int"},
        func=add,
    )
)
registry.register(
    Tool(
        name="multiply",
        description="Multiply two numbers",
        input_schema=ToolMultiplyArgs,
        output_schema={"result": "int"},
        func=multiply,
    )
)

In [15]:
# Get type-safe tool names and arguments
ToolNameLiteral = registry.get_tool_names()
ToolArgsUnion = registry.get_tool_call_args_type()

class ToolCall(BaseModel):
    action: Literal["tool"]
    thought: str
    tool_name: ToolNameLiteral
    args: ToolArgsUnion
class FinalAnswer(BaseModel):
    action: Literal["final"]
    answer: str
LLMResponse = Union[ToolCall, FinalAnswer]

In [16]:
# Get type-safe tool names and arguments
ToolNameLiteral = registry.get_tool_names()
ToolArgsUnion = registry.get_tool_call_args_type()

class ToolCall(BaseModel):
    action: Literal["tool"]
    thought: str
    tool_name: ToolNameLiteral
    args: ToolArgsUnion
class FinalAnswer(BaseModel):
    action: Literal["final"]
    answer: str
LLMResponse = Union[ToolCall, FinalAnswer]

In [17]:
def _create_system_instruction(self) -> str:
    tools_description = json.dumps(
        self.tool_registry.list_tools(),
        indent=2
    )

    system_prompt = """
You are a conversational AI agent that can interact with external tools.
CRITICAL RULES (MUST FOLLOW):
- You are NOT allowed to perform operations internally that could be performed by an available tool.
- If a tool exists that can perform any part of the task, you MUST use that tool.
- You MUST NOT skip tools, even for simple or obvious steps.
- You MUST NOT combine multiple operations into a single step unless a tool explicitly supports it.
- You may ONLY produce a final answer when no available tool can further advance the task.
TOOL USAGE RULES:
- Each tool call must perform exactly ONE meaningful operation.
- If the task requires multiple operations, you MUST call tools sequentially.
- If multiple tools could apply, choose the most specific one.
RESPONSE FORMAT (STRICT):
- You MUST respond ONLY in valid JSON.
- Never include explanations outside JSON.
- You must choose exactly one action per response.
Tool call format:
{
  "action": "tool",
  "thought": "...",
  "tool_name": "...",
  "inputs": { ... }
}
Final answer format:
{
  "action": "final",
  "answer": "..."
}""" + "\\n\\nAvailable tools with description:\\n" + tools_description
    return system_prompt

In [18]:
def _format_gemini_chat_history(self, history: list[dict]) -> list:
    formatted_history = []
    for message in history:
        if message["role"] == "user":
            formatted_history.append(types.Content(
                    role="user",
                    parts=[
                        types.Part.from_text(text=message["content"])
                    ]
                )
            )
        if message["role"] == "assistant":
            formatted_history.append(types.Content(
                    role="model",
                    parts=[
                        types.Part.from_text(text=message["content"])
                    ]
                )
            )
        if message["role"] == "tool":
            formatted_history.append(types.Content(
                    role="tool",
                    parts=[
                        types.Part.from_function_response(
                            name=message["tool_name"],
                            response={'result': message["tool_response"]},
                        )
                    ]
                )
            )
    return formatted_history

In [19]:
def generate(self, history: list[dict]) -> str:
    gemini_history_format = self._format_gemini_chat_history(history)
    response = self.client.models.generate_content(
        model=self.model,
        contents=gemini_history_format,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=LLMResponse,
            system_instruction=self.system_instruction,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
        ),
    )
    return response.text

In [20]:
def generate(self, history: list[dict]) -> str:
    gemini_history_format = self._format_gemini_chat_history(history)
    response = self.client.models.generate_content(
        model=self.model,
        contents=gemini_history_format,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=LLMResponse,
            system_instruction=self.system_instruction,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
        ),
    )
    return response.text

In [21]:
def run(self, user_input: str):
    self.history.append({"role": "user", "content": user_input})
    for step in range(self.max_steps):
        # Get LLM decision
        llm_output = self.llm.generate(self.history)
        action = json.loads(llm_output)
        if action["action"] == "tool":
            # Record the thought process
            self.history.append(
                {"role": "assistant", "content": llm_output}
            )
            # Execute the tool
            tool = self.tool_registry.get(action["tool_name"])
            result = tool(**action["args"])
            # Record the result
            observation = f"Tool {tool.name} returned: {result}"
            self.history.append(
                {"role": "tool", "tool_name": tool.name, "tool_response": result}
            )
            continue
        if action["action"] == "final":
            self.history.append(
                {"role": "assistant", "content": llm_output}
            )
            return action["answer"]
    raise RuntimeError("Agent did not terminate within max_steps")

In [22]:
def run(self, user_input: str):
    self.history.append({"role": "user", "content": user_input})
    for step in range(self.max_steps):
        # Get LLM decision
        llm_output = self.llm.generate(self.history)
        action = json.loads(llm_output)
        if action["action"] == "tool":
            # Record the thought process
            self.history.append(
                {"role": "assistant", "content": llm_output}
            )
            # Execute the tool
            tool = self.tool_registry.get(action["tool_name"])
            result = tool(**action["args"])
            # Record the result
            observation = f"Tool {tool.name} returned: {result}"
            self.history.append(
                {"role": "tool", "tool_name": tool.name, "tool_response": result}
            )
            continue
        if action["action"] == "final":
            self.history.append(
                {"role": "assistant", "content": llm_output}
            )
            return action["answer"]
    raise RuntimeError("Agent did not terminate within max_steps")

In [24]:
{
  "action": "tool",
  "thought": "I need to first add 5 and 3",
  "tool_name": "add",
  "args": {"a": 5, "b": 3}
}

{'action': 'tool',
 'thought': 'I need to first add 5 and 3',
 'tool_name': 'add',
 'args': {'a': 5, 'b': 3}}